# Task 3: Advanced optimization techniques 

In this unit we will focus on some techniques that help solving complex optimization problems. You will learn how to warm start the solver, how to make use from symmetry breaking constraints and how to tackle multi-criteria problems. 
For the analysis we will refer to an extended version of the problem from the last unit.

In [ ]:
# First you can check, whether the required packages are already installed; if not, you'll receive a warning
# Use the `%` symbol to run shell commands directly in the notebook.
%pip show gurobipy
%pip show matplotlib
%pip show numpy
%pip show pandas

In [ ]:
# Install required packages
# %pip install gurobipy
# %pip install matplotlib
# %pip install numpy
# %pip install pandas

Import everything:

In [ ]:
# importing the required packages
import gurobipy as gp # package as interface for Gurobi
from gurobipy import GRB # Gurobi environment
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

## Part 1: Extended model 
To see the effect of performance improvement measures, let's extend the model formulation to a multiple machine case.

### Data

In [ ]:
# Import the extended .csv file, sort, and save into DataFrame 
df = pd.read_csv("data/WorkTasksExtended_1.csv", parse_dates=['Due date'], sep=';')
df = df.rename(columns={'No.': 'no', 'Due date': 'due_date', 'Type': 'type'})  # rename coloumns

df.head(10)

>Check the size. What do you notice as you reflect on last week's modelling excercise?

In [ ]:
df.shape

In [ ]:
# Here comes the data; please note that we add two more resources

# Data input for mathematical program
d_all = np.arange('2020-05-11', '2020-06-01', dtype='datetime64[D]') # initialize planning horizon of 20 days: array of periods within the given range in steps of one day

T_plan = 20; # define length of plan

# DataFrame of tasks for optimization; 1st day is May 11
df_optimize = df[df["due_date"] >= np.datetime64('2020-05-11') ]
df_optimize = df_optimize[df_optimize["due_date"] <= np.datetime64('2020-05-11') + T_plan - 1 ]
df_optimize = df_optimize.reset_index(drop=True) # Reset the index numbers of the new df

types = sorted(df_optimize['type'].unique()) # create array of unique product types

num_types = len(types) # calculate number of product types
num_tasks = len(df_optimize) # calculate number of tasks (= number of elements in 1st column of DataFrame)

cap = 30; # initialize capacity per resource
d_task = df_optimize['due_date'].values # create array of due dates (= 2nd column in DataFrame)
n_task = df_optimize['no'].values # create array of task number (= 1st column in DataFrame)

# here comes the extension for the three machine case
n_resource = [1, 2, 3] # create array of resources 
num_resources = len(n_resource) # calculate number of resources

In [ ]:
# Define new array of binaries to indicate whether each task is of a certain type i
task_types= np.zeros((num_tasks, num_types), dtype=bool).astype(int) # initialize (num_task x num_type) array 

for i in range(num_types):
    task_types[:,i] = (df_optimize['type'].values == types[i]) # will add a 1 (=true) to the ith column for each task, if type of task (3rd column) is i, else 0

print(task_types)

## Model

We formulate a MILP to minimize lateness (positive and negative violations) for the defined length of the plan of one week.

>Inspect the model for changes with respect to last week's model. There should be six of them. 


>How will the extensions effect the solution time? Solve the model to check out. So, start the optimization and maybe grab a coffee.

In [ ]:
# Initialize the model
MILP = gp.Model("MILP")

# Variables
X = MILP.addVars(T_plan, num_tasks, num_resources, vtype=GRB.BINARY, name="X") # initialize binary assignment variables (task --> period)
DueDev = MILP.addVars(num_tasks, lb=0, name="DueDev") # initialize lateness variable

# Objective function: minimize total lateness
MILP.setObjective(sum(DueDev[i] for i in range(num_tasks)), GRB.MINIMIZE)

# Add constraints
Earliness = MILP.addConstrs((DueDev[i] >= sum((int((d_task[i]-d_all[t]) / np.timedelta64(1, 'D')))*X[t,i,m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Earliness')
Tardiness = MILP.addConstrs((DueDev[i] >= sum((int((d_all[t]-d_task[i]) / np.timedelta64(1, 'D')))*X[t,i,m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Tardiness')
CapacityRestriction = MILP.addConstrs((sum( X[t,i,m] for i in range(num_tasks)) <= cap for t in range(T_plan) for m in range(num_resources)), name='CapacityRestriction')
TaskExactlyOnce = MILP.addConstrs((sum(X[t,i,m] for t in range(T_plan) for m in range(num_resources)) == 1 for i in range(num_tasks)), name='TaskExactlyOnce')
OnlyOneHard = MILP.addConstrs((sum( X[t,i,m]*task_types[i,0] for i in range(num_tasks)) <= 1 for t in range(T_plan) for m in range(num_resources)), name='OnlyOneHard')

# Suppress console output
MILP.setParam('OutputFlag', 0)

# Fraction of time spent on heuristics
MILP.setParam('Heuristics', 0)


# Optimize model
MILP.optimize()

## Results

In [ ]:
print("Objective value: ", MILP.objVal, " found after ", MILP.Runtime, " seconds. Relative gap is: ", MILP.MIPGap)

In [ ]:
# set length of interval for the analysis
T_analyze=20

In [ ]:
# Initialize results array with None values
X_results = np.empty(shape=(num_resources, T_analyze, cap), dtype='object')
X_results.fill(None)

# Parse the solution
for m in range(num_resources):
    print(f'Machine {m+1}')
    for period in range(T_analyze):
        print('day', period, ':  ', end='')
        i = 0  # counter for the number of tasks assigned in this period and machine
        for task in range(num_tasks):
            if X[period, task, m].X > 0:
                if i < cap:  # avoid index out of bounds
                    X_results[m, period, i] = n_task[task]
                    print(n_task[task], '\t', end='')
                    i += 1
        print("")

## Adjusting Gurobi´s parameters

As you may have noticed, the model above has in comparison to last week an additional parameter. 

    MILP.setParam('Heuristics', 0)

This parameter completely deactivates Gurobi's heuristics. Typically, these heuristics play a key role in quickly finding good feasible solutions early in the optimization process. This can greatly reduce computation time and make the solving process more efficient overall.

Its functionality is somewhat akin to that of a warm start. What this means and what happens in the process will be explained in more detail shortly with an example. The fact that the use of heuristics can have a massive impact on runtime is already evident here through the adjustment of the heuristics parameter.   

By default, the heuristic parameter is set to 0.05 – meaning Gurobi may spend up to 5% of its time on heuristic methods. This value can be set anywhere from 0 (off) to 1 (maximum heuristic effort).

    MILP.Params.Heuristics = 0.1 # What is happening? 
    MILP.Params.Heuristics = 0.5 # What is happening? 

Go back to the model formulation and try different values for the heuristics parameter! What can you notice after changing the heuristics parameters to 0.1 and 0.5?

## Adding solver parameters

In case of excessive runtimes we might need to control the runtime of the solver. While helpful in practice, this comes at the risk of not finding the optimal solution.

The `setParam` function is used to set the solver parameters. You may set different parameters depending on the specific needs of your problem and your performance requirements.

Some parameter examples for Gurobipy ([Gurobi](https://www.gurobi.com/documentation/9.1/refman/parameters.html)):
```python
# Set a gap for the objective function value of the best solution found and the (theoretical) bound 
# The solution will stop when it is not larger than 0.5% of upper bound 
# (solution is at worst 0.5% worse than the optimal one)
m.setParam('MIPGap', 0.05)  # 0.5% gap

# Limit on solution search time in seconds (feasible solution and solution quality are not guaranteed)
m.setParam('TimeLimit', 60)  # 60 seconds

# No text output 
m.setParam('OutputFlag', 0)  
```

>Go back to the initial model formulation and give it a try. What do you notice as you reduce the time limit? 

### Plots

In [ ]:
# Iterate through each resource
for m in range(num_resources):
    # Create a figure and a set of subplots
    fig, ax = plt.subplots() 
    
    # Get number of days
    num_days = X_results.shape[1] # .shape[1] -> number of columns

    # Iterate through each day
    for day in range(num_days):
        # Filter out None values
        tasks = [task for task in X_results[m, day] if task is not None]
        
        # Plot each task as a separate bar, stacking them by adjusting the bottom position
        bottom = 0
        for task in tasks:
            ax.bar(day, 1, bottom=bottom, color='lightblue', edgecolor='black', linewidth=0.5)
            # Add text
            ax.text(day, bottom + 0.5, str(task), va='center', ha='center', fontsize=6)
            bottom += 1

    # Set x ticks and labels
    ax.set_xticks(range(num_days))
    ax.set_xticklabels([day+1 for day in range(num_days)], fontsize=6)

    # Set labels
    ax.set_xlabel('Day')
    ax.set_ylabel('Tasks')
    ax.set_title(f'Tasks per Day for Machine {m+1}')

# Show the plots
plt.show()

Mission 1 accomplished 🏅 * take a break 🥐🍦 

## Part 2: Using warm start

Warm starting refers to the idea of providing the solver with a first feasible solution. While typically not optimal, the feasible solution allows the solver to more efficiently explore the solution space. 

>Inspect the following code: 
>
>- What does it do? 
>- What do we get as a result? 
>- What is your assessment of the procedure?


In [ ]:
# Setting warm start variable

X_WS = np.zeros((T_plan, num_tasks, num_resources)) # Create an empty matrix; will be used to represent the initial solution provided to the mathematical program
p = 0 # Auxiliary variable for period
i = 0 # Auxiliary variable for task
m = 0 # Auxiliary variable for machine

# what is happening here?
while i < num_tasks: 
    if p >= T_plan:
        m += 1 # Move to next machine
        p = 0 # Restart from period 0
    else:
        if task_types[i, 0] == 1: # Check for task of type 1
            X_WS[p, i, m] = 1 # Assign task to machine m
            p += 1 # Increment period
            i += 1 # Increment task
        else: 
            i += 1 # Increment task

# Step 2: Assignment of all remaining tasks
p = 0
i = 0
m = 0

while i < num_tasks:
    if p >= T_plan: 
        m += 1 
        p = 0
    else:
        if task_types[i, 0] != 1: # what is happening here?
            if np.sum(X_WS[p, :, m]) < cap: # Make sure that capacity suffices
                X_WS[p, i, m] = 1
                i += 1
            else:
                p += 1
        else:
            i += 1

print(X_WS) # Print initial assignment

Now we add the initial solution to warm start the model and start the solution procedure. Please note, how simple it is done.

In [ ]:
# Initialize the model
MILP = gp.Model("MILP")

# Variables
X = MILP.addVars(T_plan, num_tasks, num_resources, vtype=GRB.BINARY, name="X") # initialize binary assignment variables (task --> period)

# Setting warm start for all X variables
for t in range(T_plan):
    for i in range(num_tasks):
        for m in range(num_resources):
            X[t, i, m].start = X_WS[t, i, m]

DueDev = MILP.addVars(num_tasks, lb=0, name="DueDev") # initialize lateness variable

# Objective function: minimize total lateness
MILP.setObjective(sum(DueDev[i] for i in range(num_tasks)), GRB.MINIMIZE)

# Add constraints
Earliness = MILP.addConstrs((DueDev[i] >= sum((int((d_task[i]-d_all[t]) / np.timedelta64(1, 'D')))*X[t,i,m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Earliness')
Tardiness = MILP.addConstrs((DueDev[i] >= sum((int((d_all[t]-d_task[i]) / np.timedelta64(1, 'D')))*X[t,i,m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Tardiness')


CapacityRestriction = MILP.addConstrs((sum( X[t,i,m] for i in range(num_tasks)) <= cap for t in range(T_plan) for m in range(num_resources)), name='CapacityRestriction')
TaskExactlyOnce = MILP.addConstrs((sum(X[t,i,m] for t in range(T_plan) for m in range(num_resources)) == 1 for i in range(num_tasks)), name='TaskExactlyOnce')
OnlyOneHard = MILP.addConstrs((sum( X[t,i,m]*task_types[i,0] for i in range(num_tasks)) <= 1 for t in range(T_plan) for m in range(num_resources)), name='OnlyOneHard')

# Suppress console output
MILP.setParam('OutputFlag', 0)

# Fraction of time spent on heuristics
MILP.setParam('Heuristics', 0)


# Re-optimizing model with warm start; the model above still exists
MILP.optimize()

In [ ]:
print("Objective value: ", MILP.objVal, " found after ", MILP.Runtime, " seconds. Relative gap is: ", MILP.MIPGap)

>Do you notice any difference to the solution of the inital model formulation? Does the time limit have any effect?

Mission 2 accomplished 👍.

## Part 3: Symmetry breaking constraints

MILPs often suffer from symmetry. This is the case, if multiple solutions of the mathematical program refer to the same real-world solution. Accordingly, we have multiple solutions with equal objective function values. In such a situation the solver will need to explore a vast number of potential solutions in the search procedure (branch-and-bound) to prove optimality. 

Assume a vehicle routing setting where six customers (1, 2, 3, 4, 5, 6) are to be served using two identical service vehicles (A, B). Each vehicle can serve three customers. One possible solution is to travel the route 1->2->3 with A and the route  4->5->6 with B. Symmetry is a problem as (B:1->2->3, A:4->5->6) refers to the same solution as do (A:1->2->3, B:6->5->4), (B:1->2->3, A:6->5->4), ... To read more about symmetry in mathematical programs, please refer to the supplementary material provided in ISIS.

A widely used technique to break symmetry is to add feasible inequalities (=constraints), which eliminate irrelevant areas of the search space. For some problems this significantly increases solution speed. 

Let's see how it works. 

>Before we get down to business, what kind of symmetry do we face in our problem formulation?

In [ ]:
# Add symmetry-breaking variables and constraints
Util = MILP.addVars(T_plan, num_resources, lb=0, name="Util") # add new variable for the utilization of each machine in each period

UtilCalc = MILP.addConstrs((Util[t, m] == sum(X[t, i, m] for i in range(num_tasks))/cap for m in range(num_resources) for t in range(T_plan)), name='UtilCalc') # what is happening here?
Prio = MILP.addConstrs((Util[t, m] >= Util[t, m+1] for m in range(num_resources-1) for t in range(T_plan)), name='Prio') # what is happening here?

In [ ]:
# Set solver attributes
MILP.setParam("OutputFlag", 0) # no solver output

# Optimize model - the model above still exists
MILP.optimize() 

In [ ]:
# Double-check that the additional constraints work
MILP.getAttr('X', Util)

>Everything fine here?

Now, let's take a look on the results.

In [ ]:
print("Objective value: ", MILP.objVal, " found after ", MILP.Runtime, " seconds. Relative gap is: ", MILP.MIPGap)


>How does the model formulation's performance compare?

... and we are done here. One more round to go 🏃‍♀️🏃‍♂️.

- - -

## Multi-objective Decision Making: Balancing Economic and Social Objectives

In this notebook, we explore a scenario where we aim to balance economic (minimizing due date deviations) and social (minimizing Sunday work) objectives in a scheduling context. We'll explore the trade-off between these objectives, seeking an efficient frontier to aid decision-makers in making an informed choice. 

To make the next example easier to understand, we will work with a reduced version of the one above, which still extends the example from the last unit. 

In [ ]:
df = pd.read_csv("data/WorkTasksExtended_2.csv", parse_dates=['Due date'])
df = df.rename(columns={'No.': 'no', 'Due date': 'due_date', 'Type': 'type'}) 

In [ ]:
d_all = np.arange('2020-05-11', '2020-06-01', dtype='datetime64[D]') 

df_optimize = df[df["due_date"] >= np.datetime64('2020-05-11') ]
df_optimize = df_optimize[df_optimize["due_date"] <= np.datetime64('2020-05-11') + T_plan - 1 ]
df_optimize = df_optimize.reset_index(drop=True) 

types = sorted(df_optimize['type'].unique()) 

num_types = len(types) 
num_tasks = len(df_optimize) 

cap = 10; 
d_task = df_optimize['due_date'].values 
n_task = df_optimize['no'].values 


n_resource = [1, 2] 
num_resources = len(n_resource) 

In [ ]:
task_types= np.zeros((num_tasks, num_types), dtype=bool).astype(int)

for i in range(num_types):
    task_types[:,i] = (df_optimize['type'].values == types[i])

In [ ]:
# Initialize the model
MILP = gp.Model("MILP")

# Variables
X = MILP.addVars(T_plan, num_tasks, num_resources, vtype=GRB.BINARY, name="X") # initialize binary assignment variables (task --> period)
DueDev = MILP.addVars(num_tasks, lb=0, name="DueDev") # initialize lateness variable

# Objective function: minimize total lateness
MILP.setObjective(sum(DueDev[i] for i in range(num_tasks)), GRB.MINIMIZE)

# Add constraints
Earliness = MILP.addConstrs((DueDev[i] >= sum((int((d_task[i]-d_all[t]) / np.timedelta64(1, 'D')))*X[t,i,m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Earliness')
Tardiness = MILP.addConstrs((DueDev[i] >= sum((int((d_all[t]-d_task[i]) / np.timedelta64(1, 'D')))*X[t,i,m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Tardiness')
CapacityRestriction = MILP.addConstrs((sum( X[t,i,m] for i in range(num_tasks)) <= cap for t in range(T_plan) for m in range(num_resources)), name='CapacityRestriction')
TaskExactlyOnce = MILP.addConstrs((sum(X[t,i,m] for t in range(T_plan) for m in range(num_resources)) == 1 for i in range(num_tasks)), name='TaskExactlyOnce')
OnlyOneHard = MILP.addConstrs((sum( X[t,i,m]*task_types[i,0] for i in range(num_tasks)) <= 1 for t in range(T_plan) for m in range(num_resources)), name='OnlyOneHard')

# Suppress console output
MILP.setParam('OutputFlag', 0)

# Fraction of time spent on heuristics
MILP.setParam('Heuristics', 0)


# Optimize model
MILP.optimize()

In [ ]:
min_lateness = MILP.ObjVal
min_lateness  # Objective Value of the Model with the new data

### Step 1: Determine Minimum Value for Sunday Labor (Objective 2)

We first aim to find the minimum possible value for Sunday labor by modeling it as a second objective. This step involves identifying Sundays within the planning horizon and formulating a model that considers both objectives.


In [ ]:
T_plan = 20

# Identify Sundays in the planning horizon
is_sunday = np.zeros(T_plan)
is_sunday[6] = 1  # 7th period is a Sunday
is_sunday[13] = 1  # 14th period is a Sunday

print(is_sunday)

### Step 2: Model Formulation Considering Both Objectives

We extend the model formulation to accommodate the second objective (minimizing Sunday labor) while keeping track of total lateness (first objective). Let's formulate and solve this bi-objective model.


In [ ]:
# Initialize the model
MILP = gp.Model("MILP")

# Variables
X = MILP.addVars(T_plan, num_tasks, num_resources, vtype=GRB.BINARY, name="X")
DueDev = MILP.addVars(num_tasks, lb=0, name="DueDev")
Util = MILP.addVars(T_plan, num_resources, name="Util")
MaxUtil = MILP.addVar(name="MaxUtil")
TotalDev = MILP.addVar(name="TotalDev")

# Constraints
MILP.addConstrs((DueDev[i] >= sum((int((d_task[i] - d_all[t]) / np.timedelta64(1, 'D'))) * X[t, i, m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Earliness')
MILP.addConstrs((DueDev[i] >= sum((int((d_all[t] - d_task[i]) / np.timedelta64(1, 'D'))) * X[t, i, m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Tardiness')
MILP.addConstrs((sum(X[t, i, m] for i in range(num_tasks)) <= cap for t in range(T_plan) for m in range(num_resources)), name='CapacityRestriction')
MILP.addConstrs((sum(X[t, i, m] for t in range(T_plan) for m in range(num_resources)) == 1 for i in range(num_tasks)), name='TaskExactlyOnce')
MILP.addConstrs((sum(X[t, i, m] * task_types[i, 0] for i in range(num_tasks)) <= 1 for t in range(T_plan) for m in range(num_resources)), name='OnlyOneHard')
MILP.addConstrs((Util[t, m] == gp.quicksum(X[t, i, m] for i in range(num_tasks)) / cap for t in range(T_plan) for m in range(num_resources)), name="UtilCalc")
MILP.addConstrs((Util[t, m] >= Util[t, m + 1] for m in range(num_resources - 1) for t in range(T_plan)), name="Prio")
MILP.addConstr(TotalDev == gp.quicksum(DueDev[i] for i in range(num_tasks)), name="TotalDeviation")
MILP.addConstrs((MaxUtil >= Util[t, m] * is_sunday[t] for t in range(T_plan) for m in range(num_resources)), name="UMax")

# Objective function: Minimize MaxUtil
MILP.setObjective(MaxUtil, GRB.MINIMIZE)

# Suppress console output
MILP.setParam('OutputFlag', 0)

# Set time limit
MILP.setParam('TimeLimit', 360)

# Optimize model
MILP.optimize()

Let's save the optimal value for the minimum Sunday utilization.

In [ ]:
min_U = MaxUtil.X
min_U

> Take a detailed look on the results:
> 
> - What is the total deviation?
> - How does it compare?
> - How can we improve this figure... without compromising on objective 2? We will get to this in the next step, but maybe you already have an idea.


Based on your considerations, let's reconsider the results to reveal the true trade-off. This will involve a modified problem formulation, as follows:

In [ ]:
# Initialize the model
MILP = gp.Model("MILP")

# Variables
X = MILP.addVars(T_plan, num_tasks, num_resources, vtype=GRB.BINARY, name="X")
DueDev = MILP.addVars(num_tasks, lb=0, name="DueDev")
Util = MILP.addVars(T_plan, num_resources, name="Util")
TotalDev = MILP.addVar(name="TotalDev")

# Constraints
MILP.addConstrs((DueDev[i] >= sum((int((d_task[i] - d_all[t]) / np.timedelta64(1, 'D'))) * X[t, i, m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Earliness')
MILP.addConstrs((DueDev[i] >= sum((int((d_all[t] - d_task[i]) / np.timedelta64(1, 'D'))) * X[t, i, m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Tardiness')
MILP.addConstrs((sum(X[t, i, m] for i in range(num_tasks)) <= cap for t in range(T_plan) for m in range(num_resources)), name='CapacityRestriction')
MILP.addConstrs((sum(X[t, i, m] for t in range(T_plan) for m in range(num_resources)) == 1 for i in range(num_tasks)), name='TaskExactlyOnce')
MILP.addConstrs((sum(X[t, i, m] * task_types[i, 0] for i in range(num_tasks)) <= 1 for t in range(T_plan) for m in range(num_resources)), name='OnlyOneHard')
MILP.addConstrs((Util[t, m] == gp.quicksum(X[t, i, m] for i in range(num_tasks)) / cap for t in range(T_plan) for m in range(num_resources)), name="UtilCalc")
MILP.addConstrs((Util[t, m] >= Util[t, m + 1] for m in range(num_resources - 1) for t in range(T_plan)), name="Prio")
MILP.addConstr(TotalDev == gp.quicksum(DueDev[i] for i in range(num_tasks)), name="TotalDeviation")

# what is happening here?
MILP.addConstrs((Util[t, m] <= min_U for t in range(T_plan) for m in range(num_resources) if is_sunday[t]), name="UMax")

# what is happening here?
MILP.setObjective(TotalDev, GRB.MINIMIZE)

# Suppress console output
MILP.setParam('OutputFlag', 0)

# Set time limit
MILP.setParam('TimeLimit', 360)

# Optimize model
MILP.optimize()

Let's save the result for later.

In [ ]:
# Save the resultant total deviation (now possibly increased) for later analysis
max_lateness = TotalDev.X
max_lateness

We now know the optimal total lateness, AFTER optimizing for Sunday labor and we know that it is much smaller than that of the initial solution. This is another good argument for carefully analyzing the decision situation not to leave out any relevant criteria. Otherwise, you might find yourself with highly unappreciated (unbalanced) solutions.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract the results from the Gurobi model variables
X_results = np.array([[[X[t, i, m].X for m in range(num_resources)] for i in range(num_tasks)] for t in range(T_plan)])

# Generate the grouped bar chart
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Machine 1
ax[0].bar(np.arange(T_plan), X_results[:, :, 0].sum(axis=1))
ax[0].set_title("Machine 1")
ax[0].set_xticks(np.arange(T_plan))
ax[0].set_yticks(np.arange(cap+1))
ax[0].set_ylim([0, cap])

# Machine 2
ax[1].bar(np.arange(T_plan), X_results[:, :, 1].sum(axis=1))
ax[1].set_title("Machine 2")
ax[1].set_xticks(np.arange(T_plan))
ax[1].set_yticks(np.arange(cap+1))
ax[1].set_ylim([0, cap])

plt.show()


Now we know that at best we will need to complete at least three tasks on each machine on the two Sundays. This will result in a total deviation of 216.

The combination of the best values for both objective values is called the ideal point. The combination of the prevailing values of the other objective function is called the Nadir point. Refer to the following example and the supplementary reading for further details.

<img src="figures/Characteristic_points_MODM.png" alt="MODM points" width="1000">

For our example we have:

- Ideal point: Lateness: 164, Utilization: 0.4

For the Nadir point, we have to re-run the model to optimize for objective 2 subject to the following new constraint: TotalDev <= 164. It follows:

- Nadir point: Lateness: 216, Utilization: 1.0


### Step 2: Determine Efficient Frontier Using the Epsilon-constraint Approach

To determine the efficient frontier, we will use the epsilon-constraint approach. This approach iteratively solves the model and imposes a different value for the maximal allowed Sunday labor to each iteration.

<img src="figures/e_constraint.png" alt="MODM points" width="1000">

We will consider values for Sunday labor ranging from the minimal value (ideal point) to the maximal value (Nadir point) in step size of one order (= 10%). To come up with a lean model code, we will define the model as a function that is iteratively called in the solution procedure.


In [ ]:
def ProductionPlan(epsilon, runtime): # set parameters for max. allowed Sunday utilization and runtime
    MILP = gp.Model("MILP")
    
    # Variables
    X = MILP.addVars(T_plan, num_tasks, num_resources, vtype=GRB.BINARY, name="X") # initialize binary assignment variables (task --> period)
    DueDev = MILP.addVars(num_tasks, lb=0, name="DueDev") # initialize lateness variable
    Util = MILP.addVars(T_plan, num_resources, lb=0, name="Util")
    MaxUtil = MILP.addVar(lb=0, name="MaxUtil")
    TotalDev = MILP.addVar(lb=0, name="TotalDev")

    # Objective
    MILP.setObjective(TotalDev, GRB.MINIMIZE) # objective function: minimize total lateness

    # Constraints
    UtilCalc = MILP.addConstrs((Util[t, m] == gp.quicksum(X[t, i, m] for i in range(num_tasks))/cap for m in range(num_resources) for t in range(T_plan)), name='UtilCalc')
    UMax = MILP.addConstrs((MaxUtil >= Util[t, m] for m in range(num_resources) for t in range(T_plan) if is_sunday[t] == 1), name="UMax")
    ULimit = MILP.addConstr(MaxUtil <= epsilon, name="ULimit")
    Prio = MILP.addConstrs((Util[t, m] >= Util[t, m + 1] for m in range(num_resources - 1) for t in range(T_plan)), name="Prio")
    Earliness = MILP.addConstrs((DueDev[i] >= gp.quicksum((int((d_task[i] - d_all[t]) / np.timedelta64(1, 'D'))) * X[t, i, m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Earliness')
    Tardiness = MILP.addConstrs((DueDev[i] >= gp.quicksum((int((d_all[t] - d_task[i]) / np.timedelta64(1, 'D'))) * X[t, i, m] for t in range(T_plan) for m in range(num_resources)) for i in range(num_tasks)), name='Tardiness')
    CapacityRestriction = MILP.addConstrs((gp.quicksum(X[t, i, m] for i in range(num_tasks)) <= cap for t in range(T_plan) for m in range(num_resources)), name="CapacityRestriction")
    TaskExactlyOnce = MILP.addConstrs((gp.quicksum(X[t, i, m] for t in range(T_plan) for m in range(num_resources)) == 1 for i in range(num_tasks)), name="TaskExactlyOnce")
    OnlyOneHard = MILP.addConstrs((gp.quicksum(X[t, i, m] * task_types[i, 0] for i in range(num_tasks)) <= 1 for t in range(T_plan) for m in range(num_resources)), name="OnlyOneHard")


    TotalDeviation = MILP.addConstr(TotalDev == gp.quicksum(DueDev[i] for i in range(num_tasks)), name="TotalDeviation") # total lateness
    MILP.setParam('OutputFlag', 0) # no text output
    MILP.setParam('TimeLimit', runtime) # limit on solution search time [s]
    MILP.optimize()

    return TotalDev.X, MaxUtil.X, MILP.MIPGap

# Given parameters
u_lbound = min_U * 100 # lower boundary for optimal value for the maximum Sunday utilization [percent]
u_ubound = 100 # upper boundary for maximal utilization of Sunday [percent]
stepsize = 10 # stepsize for iteration [percentage points]
steps = int((u_ubound - u_lbound) / stepsize + 1) # determine resulting number of steps (=iterations)
timelim = 360 # timelimit for each iteration

TL = np.zeros(steps) # initialize array to save results: total lateness (objective 1)
U = np.zeros(steps) # initialize array to save results: resulting maximum utilization of Sunday of iteration (objective 2)
Gap = np.zeros(steps) # MIP gap of iteration (could be >0 due to time limit on solution time)

i = 0 # counter for iteration
epsilon = u_lbound # auxiliary variable for maximum allowed utilization

while epsilon <= u_ubound: # solve model for each value of epsilon
    TL[i], U[i], Gap[i] = ProductionPlan(epsilon / 100, timelim)
    i += 1
    epsilon += stepsize

Well done 🏁

Almost done. Let's pick up the bread crumbs and depict the results.

In [ ]:
import matplotlib.pyplot as plt

max_U = 1 

# Create scatter plots
plt.scatter(TL, U, color='blue', label='Non-dominated solutions')
plt.scatter([min_lateness], [min_U], color='green', label='Ideal point') # combination of best objective function values
plt.scatter([max_lateness], [max_U], color='red', label='Nadir point') # combination of objective function values after optimizing for the other objective function (e.g., minimum utilization that can be achived, if optimizing for total lateness)

# Set labels and legend
plt.ylabel('Maximum Sunday utilization')
plt.xlabel('Total lateness')
plt.legend(loc='upper right')

# Customize ticks on the axes
plt.xticks(range(160, 220, 10))
plt.yticks([i/10 for i in range(4, 11)])
plt.legend(bbox_to_anchor=(1, 0.9))

# Show plot
plt.show()

What a beautiful efficient frontier, isn't it. Which solution would you go with and why?

All set. Next level: capacity planning. See you around. ✨